# 🍽️ Notebook 1 · Restaurant — Class Design

Welcome! In this lab we design a small **Restaurant Point-of-Sale (POS)** system
using object-oriented design. A POS is the software a waiter uses to:

- show the menu to guests,
- seat people at tables,
- take an order,
- send it to the kitchen,
- print the bill at the end.

We'll follow the repo pattern: **bad first, then best**, so you can *see* why
each design choice matters.

> If you've never designed a class before, don't worry. We'll build up slowly.


## 🛠️ Setup

```bash
cd 07-object-oriented-design/restaurant
uv sync
```

Then pick the `.venv` kernel in VS Code (top-right of the notebook). If it
doesn't show up: `Cmd+Shift+P` → **Reload Window**.


## 1️⃣ The problem

Imagine you're opening a pizzeria. Customers walk in, sit at a table, order
food, eat, and pay. Your job is to write the *software model* — the Python
classes that represent all of that.

Before we think about Python, let's list the **nouns** and **verbs** from the
story above. Nouns tend to become classes; verbs tend to become methods.

| Nouns (classes?) | Verbs (methods?) |
|---|---|
| Menu, MenuItem | show menu, find item |
| Table | seat guests, free table, reserve |
| Order, OrderItem | add item, place, serve, pay |
| Bill | compute total with tax and tip |
| Waiter, Chef, Manager | take order, cook, supervise |
| Reservation | book a table for later |

That table is literally the class diagram we're about to build. 🎯


## 2️⃣ ❌ Bad design — the God Class

The **first instinct** of a new programmer is to put everything in one class.
Let's do that on purpose so we feel the pain.


In [1]:
# ❌ DO NOT COPY THIS STYLE — it's intentionally bad.
class RestaurantSystem:
    """One class that does EVERYTHING. Hard to read, hard to test."""

    def __init__(self):
        # Menu, tables, orders — all mashed together as raw dicts/lists.
        self.menu = {"Margherita": 10, "Coke": 3, "Tiramisu": 6}
        self.tables = {1: "free", 2: "free", 3: "free"}
        self.orders = {}  # table_number -> list of (item, qty)
        self.order_status = {}  # table_number -> "open"/"placed"/"served"/"paid"

    def seat(self, table):
        if self.tables[table] != "free":
            raise ValueError("taken")
        self.tables[table] = "taken"
        self.orders[table] = []
        self.order_status[table] = "open"

    def add(self, table, item, qty=1):
        # No check that the order is still open. No check the item exists.
        self.orders[table].append((item, qty))

    def place(self, table):
        self.order_status[table] = "placed"  # no guard on current state

    def serve(self, table):
        self.order_status[table] = "served"

    def pay(self, table):
        # Tax + tip logic lives here. If we ever want a loyalty discount we
        # will have to edit this god method — that's the whole problem.
        total = sum(self.menu[i] * q for i, q in self.orders[table])
        total = total * 1.08 + total * 0.15
        self.order_status[table] = "paid"
        self.tables[table] = "free"
        return round(total, 2)


pos = RestaurantSystem()
pos.seat(1)
pos.add(1, "Margherita")
pos.add(1, "Coke", 2)
pos.place(1)
pos.serve(1)
print("God-class bill:", pos.pay(1))


God-class bill: 19.68


### Why is this bad?

Read the code again. Count the problems:

1. **One class, many jobs.** It knows about menus *and* tables *and* money
   *and* order state. A change to any one of these touches the same file.
   (Violates the **Single Responsibility Principle**.)
2. **No domain objects.** `"Margherita"` is a raw string, `10` is a raw float.
   You can't ask a menu item *"are you a dessert?"* — the info isn't there.
3. **Invalid states are possible.** `place(1)` works even if table 1 was never
   seated. Nothing protects us.
4. **No way to extend.** Want a happy-hour discount? A vegan flag? A second
   menu for brunch? You're rewriting the god class every time.
5. **Impossible to test in isolation.** You can't test "billing math" without
   also dragging in tables and orders.

We'll fix every one of these in the "best" version.


## 3️⃣ ✅ Best design — small classes with one job each

Here is the class map we'll implement in Notebook 2.

```
┌──────────┐ contains ┌──────────┐
│   Menu   │─────────▶│ MenuItem │     (name, price, category)
└──────────┘          └──────────┘
                            ▲ refers to
                            │
┌──────────┐ owns  ┌────────────┐
│  Order   │──────▶│ OrderItem  │     (menu_item, qty, notes)
└──────────┘       └────────────┘
     │ on
     ▼
┌──────────┐
│  Table   │  (number, seats, state)
└──────────┘
     ▲ handled by
     │
┌──────────┐   ┌───────┐   ┌───────────┐
│  Waiter  │   │ Chef  │   │  Manager  │   all inherit from Staff
└──────────┘   └───────┘   └───────────┘
```

Each class has **one** job. A `Table` knows about seats and state. A `Menu`
knows about items. An `Order` knows the lifecycle of one group's meal.

### Why enums instead of strings?

Raw strings (`"placed"`, `"paid"`) let typos pass silently — `"plced"` compiles
just fine. An **Enum** (like `OrderStatus.PLACED`) gives you auto-complete and
fails loudly on typos. That tiny investment prevents a whole class of bugs.

```
OrderStatus:   OPEN ─▶ PLACED ─▶ SERVED ─▶ PAID
TableState:    FREE ─▶ TAKEN ─▶ FREE
               FREE ─▶ RESERVED ─▶ TAKEN ─▶ FREE
```

Transitions go **one way**. A paid order can't jump back to open. We'll
enforce that with a tiny guard method in Notebook 2.


## 4️⃣ SOLID checklist (beginner view)

You don't need to memorise the acronym, but here's how our design scores:

| Letter | Principle | How the restaurant uses it |
|---|---|---|
| **S** | Single Responsibility | Each class has one job (`Menu` lists food, `Bill` adds up money). |
| **O** | Open/Closed | Adding a new `Staff` role (e.g. `Sommelier`) doesn't change existing code — just subclass. |
| **L** | Liskov Substitution | Any `Staff` can be used where `Staff` is expected. |
| **I** | Interface Segregation | `Order` exposes only lifecycle methods (`place`, `serve`, `pay`), not everything. |
| **D** | Dependency Inversion | `Order` depends on abstract `PricingStrategy` (Notebook 3), not a concrete tax-and-tip formula. |

### What's next?

- **Notebook 2** — turn this design into runnable Python with tests.
- **Notebook 3** — add the three OOD patterns interviewers love: **Strategy**
  (swap pricing rules), **Factory** (build menus from config), **Observer**
  (notify kitchen/manager when things happen).
